# Hype / Sentiment Index Prototype (GDELT)

Implements a free-data prototype of the HLPPL paper's "H" (Hype Index) and
"S" (Sentiment Score) components (arXiv:2510.10878, Section 3.4-3.5):

- **Hype Index**: `H_i,t = N_i,t / N_mkt,t` — stock `i`'s share of news
  attention on day `t`, relative to a reference universe's total.
- **Cap-adjusted Hype**: `CapH_i,t = H_i,t / W_cap_i,t`, where `W_cap_i,t`
  is stock `i`'s share of the reference universe's total market cap.
  `CapH > 1` = excess hype relative to economic size.
- **Sentiment Score**: `S_i,t` — daily tone of news coverage, in `[-1, 1]`.

## Data source: GDELT (free, no API key)

The [GDELT Project](https://www.gdeltproject.org/) DOC 2.0 API provides
`mode=timelinevol` (daily news-volume intensity: the fraction of all
worldwide GDELT-monitored articles mentioning a query, on each day) and
`mode=timelinetone` (daily average tone of those articles, roughly in
`[-10, 10]`).

**Important limitation**: with no `startdatetime`/`enddatetime` parameters,
the free DOC API returns a **rolling ~90-day window ending today** — it does
NOT serve arbitrary historical date ranges (explicit historical dates, e.g.
2021, returned persistent `429` errors when tested). So this prototype is a
**live/forward-looking** signal, not a historical backtest. Backtesting the
dot-com, GFC, or GME 2021 episodes with hype/sentiment features would require
GDELT's BigQuery archive (free GCP tier, full GKG history from 2015) — left
as future work (see `context/progress-tracker.md`).

## Reference universe

A small fixed basket (AAPL, MSFT, TSLA, GME) used as a placeholder reference
universe, with GameStop (GME) as the case-study target (continuing the GME
thread from `lppl_fitting.ipynb`). GDELT's free DOC API is heavily
rate-limited (observed: well under one request per minute from some network
environments), so this prototype intentionally uses a small universe to keep
the fetch tractable. A full S&P 100/500 reference universe is future work
once the pipeline mechanics are validated — see
`context/progress-tracker.md` for the rate-limit finding.

## Computing H without raw article counts

GDELT's "Volume Intensity" for ticker `i` on day `t` is
`intensity_i,t = N_i,t / N_total,t`, where `N_total,t` is the worldwide
article count (unknown to us, but common to every query). Since

```
H_i,t = N_i,t / N_mkt,t = intensity_i,t / sum_j(intensity_j,t)
```

the unknown `N_total,t` cancels — we can compute `H_i,t` directly from GDELT
intensities without ever knowing raw article counts.

In [ ]:
import json
import os
import time
import urllib.parse
import urllib.request
import urllib.error

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [ ]:
# User Inputs
try:
    from google.colab import drive
    drive.mount('/content/drive')
    project_folder = '/content/drive/MyDrive/EquityBubbleRegimes'
    if not os.path.exists(project_folder):
        os.makedirs(project_folder)
except ImportError:
    project_folder = '.'

cache_dir = os.path.join(project_folder, 'gdelt_cache')
os.makedirs(cache_dir, exist_ok=True)
print(f'Project folder: {project_folder}')
print(f'GDELT cache dir: {cache_dir}')

## 1. GDELT Fetch Function

Caches each (query, mode) timeline to disk — both to avoid re-hitting GDELT's
rate limit (the free DOC API rejects requests faster than roughly one every
~10-15 seconds with `429 Too Many Requests`) and so reruns are instant.

In [ ]:
GDELT_BASE = 'https://api.gdeltproject.org/api/v2/doc/doc'


def fetch_gdelt_timeline(query, mode, retries=10, wait=12):
    """Fetch (and cache) a GDELT DOC 2.0 timeline (timelinevol / timelinetone)
    for a query string. Returns the parsed JSON response."""
    safe_name = query.replace(' ', '_').replace('.', '')
    cache_path = os.path.join(cache_dir, f'{safe_name}_{mode}.json')

    if os.path.exists(cache_path):
        with open(cache_path) as f:
            return json.load(f)

    params = {'query': query, 'mode': mode, 'format': 'json'}
    url = GDELT_BASE + '?' + urllib.parse.urlencode(params)

    for attempt in range(1, retries + 1):
        time.sleep(wait)
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        try:
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read())
            with open(cache_path, 'w') as f:
                json.dump(data, f)
            return data
        except urllib.error.HTTPError as e:
            if e.code == 429:
                continue
            raise

    raise RuntimeError(f'GDELT request failed after {retries} attempts: {query} {mode}')


def timeline_to_series(raw, value_name):
    """Parse a GDELT timeline JSON response into a pandas Series indexed by date."""
    points = raw['timeline'][0]['data']
    dates = pd.to_datetime([p['date'] for p in points])
    values = [p['value'] for p in points]
    return pd.Series(values, index=dates, name=value_name)

In [ ]:
# Sanity check on a single ticker before the full loop
_raw_vol = fetch_gdelt_timeline('GameStop', 'timelinevol')
_vol = timeline_to_series(_raw_vol, 'volume_intensity')
print(_vol.tail())
print(f'{len(_vol)} days, {_vol.index.min().date()} to {_vol.index.max().date()}')

## 2. Reference Universe

A small fixed sample basket plus GME as the case-study target (see note on
GDELT rate limits above for why this is small).

In [ ]:
REFERENCE_UNIVERSE = {
    'AAPL': 'Apple Inc',
    'MSFT': 'Microsoft',
    'TSLA': 'Tesla Inc',
    'GME': 'GameStop',
}

TARGET = 'GME'
print(f'Reference universe: {len(REFERENCE_UNIVERSE)} tickers, target = {TARGET}')

## 3. Fetch Volume & Tone Timelines

Loops over the reference universe, fetching `timelinevol` (-> Hype numerator)
and `timelinetone` (-> Sentiment) for each ticker. Cached, so reruns are
instant.

In [ ]:
volume = {}
tone = {}

for ticker, query in REFERENCE_UNIVERSE.items():
    raw_vol = fetch_gdelt_timeline(query, 'timelinevol')
    volume[ticker] = timeline_to_series(raw_vol, ticker)

    raw_tone = fetch_gdelt_timeline(query, 'timelinetone')
    tone[ticker] = timeline_to_series(raw_tone, ticker)

    print(f'{ticker}: {len(volume[ticker])} days volume, {len(tone[ticker])} days tone')

volume_df = pd.DataFrame(volume).sort_index()
tone_df = pd.DataFrame(tone).sort_index()
volume_df = volume_df.dropna(how='any')
tone_df = tone_df.reindex(volume_df.index)
print(volume_df.shape, tone_df.shape)
volume_df.tail()

## 4. Hype Index

`H_i,t = intensity_i,t / sum_j(intensity_j,t)` over the reference universe.

In [ ]:
N_mkt = volume_df.sum(axis=1)
H = volume_df[TARGET] / N_mkt
print(H.describe())
H.tail()

## 5. Cap-Adjusted Hype Index

`CapH_i,t = H_i,t / W_cap_i,t`, where `W_cap_i,t` is the target's share of
the reference universe's total market cap.

**Simplification**: `yfinance` doesn't provide historical daily market caps
for free, so this prototype uses each ticker's *current* market cap as a
constant weight over the (short, ~90-day) window. This is a reasonable
approximation for the prototype's short window but would need point-in-time
market caps for a real backtest (documented as a known limitation).

In [ ]:
market_caps = {}
for ticker in REFERENCE_UNIVERSE:
    info = yf.Ticker(ticker).fast_info
    market_caps[ticker] = info['marketCap']

market_caps = pd.Series(market_caps)
W_cap = market_caps / market_caps.sum()
print(W_cap.sort_values(ascending=False))

CapH = H / W_cap[TARGET]
print(CapH.describe())

## 6. Sentiment Score

GDELT's average tone is roughly in `[-10, 10]` in practice (its theoretical
range is `[-100, 100]`, but extreme values are rare). Rescale to `[-1, 1]`
by dividing by 10 and clipping.

In [ ]:
S = (tone_df[TARGET] / 10.0).clip(-1, 1)
print(S.describe())

## 7. Validate

Plot `H`, `CapH`, and `S` over the ~90-day window.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(H.index, H.values)
axes[0].set_title(f'Hype Index H_{TARGET},t')
axes[0].set_ylabel('H')

axes[1].plot(CapH.index, CapH.values)
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)
axes[1].set_title(f'Cap-Adjusted Hype CapH_{TARGET},t (1.0 = hype matches market-cap weight)')
axes[1].set_ylabel('CapH')

axes[2].plot(S.index, S.values)
axes[2].axhline(0.0, color='gray', linestyle='--', linewidth=1)
axes[2].set_title(f'Sentiment Score S_{TARGET},t')
axes[2].set_ylabel('S')

plt.tight_layout()
plt.show()

## 8. Save Output

In [ ]:
output = pd.DataFrame({
    'N_i': volume_df[TARGET],
    'N_mkt': N_mkt,
    'H': H,
    'CapH': CapH,
    'S': S,
})
output.index.name = 'date'

output_path = f'{project_folder}/EquityBubbleRegimes_HypeSentiment_GME_recent90d.csv'
output.to_csv(output_path)
print(f'Saved {len(output)} rows to {output_path}')
output.tail()

## Discussion / Limitations

- **Rolling window only**: this prototype reflects roughly the last 90 days
  (relative to whenever it's run), not a fixed historical period. It cannot
  currently produce hype/sentiment features for the dot-com, GFC, or GME 2021
  case studies used in `lppl_fitting.ipynb`. A full historical version would
  use GDELT's BigQuery GKG archive (free GCP tier, 2015-present).
- **Small reference universe**: ~15 large-cap tickers is a placeholder for
  the paper's S&P 100/500-style reference universe. `H_i,t`/`CapH_i,t` values
  would shift if the universe were expanded — the *shape* of the time series
  (relative attention over time) is more meaningful than the absolute level
  at this stage.
- **Static market-cap weights**: `CapH` uses each ticker's *current* market
  cap as a constant weight across the whole window, since free historical
  daily market caps aren't readily available. Fine for a ~90-day window;
  would need point-in-time caps for a longer backtest.
- **Tone as a sentiment proxy**: GDELT's average article tone substitutes for
  the paper's FinBERT-based `S_i,t`. It's a coarser, unsupervised signal (no
  positive/neutral/negative classification, no confidence weighting) but
  requires no model training/inference — directly addresses the
  "NLP is too complicated" concern.
- **Next step**: combine `H`, `CapH`, `S` with the LPPL residual
  `ε_norm(t)` from `lppl_fitting.ipynb` into a Bubble Score, per
  `context/ai-workflow-rules.md`.